# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 34.9 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 2.5 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

# login

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.utils import shuffle
import os
import re
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import emoji
import pyarabic.araby as araby

In [ ]:
import pandas as pd

In [ ]:
import torch
import torch.distributed as dist

# Load Model

In [ ]:
model_name = "ALLaM-AI/ALLaM-7B-Instruct-preview"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )

# Assign pad_token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=20,
                temperature=0.2
               )

# Zero Shot

In [ ]:
import pandas as pd
data = pd.read_excel('News Classification Jais Zero Shot.xlsx')

In [ ]:
data.shape

(1000, 1)

# Predict Without Fine-tuning

In [ ]:
from tqdm import tqdm
max_length = tokenizer.model_max_length
for i in tqdm(range(len(data[:6]))):
    prompt = data.iloc[i]["prompt"]
    prompt = prompt[:max_length]
    result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("الفئة المتوقعة:")[-1].lower()
    print('\n')
    print(result)
    print('\n')
    print(answer)

 17%|█▋        | 1/6 [00:01<00:07,  1.54s/it]



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\n محتوى المقال: \nاستقبل الوسط المسرحي الإماراتي فوز الإمارات برئاسة الهيئة الدولية للمسرح بالكثير من الزهو والسعادة، وأعرب عدد من المشتغلين في المسرح عن فرحهم بهذا الإنجاز الذي يضاف إلى الرصيد الفني المحلي الذي قام على جهود عناصر مسرحية مسكونة بالجدية والمسؤولية .وأشار المسرحيون الإماراتيون إلى الجهود المتواصلة التي بذلها زميلهم محمد سيف الأفخم محلياً وعربياً ودولياً إلى أن قطفت الإمارات هذا النجاح الذي تستحقه، وتفوّقت على ست دول ذات رصيد تاريخي في فن المسرح .وقال د . حبيب غلوم: "نبارك لمحمد الأفخم ولأنفسنا اختياره رئيساً للهيئة الدولية للمسرح، وهو ولا شك ثمرة لاجتهاد وعطاء قيم، ونتمنى أن يترجم طموح من اختاروه، وأن تخطو به الإمارات نح

 33%|███▎      | 2/6 [00:02<00:04,  1.18s/it]



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\n محتوى المقال: \nاستضاف مركز الشارقة لفن الخط العربي والزخرفة مساء أمس الأول معرض ( حرف ولون ) لأعمال منتسبي المركز التابع لمجمع الشارقة للآداب والفنون، بحضور هنا سيف السويدي رئيسة هيئة البيئة والمحميات الطبيعية وعضو المجلس التنفيذي، وهشام المظلوم رئيس المجمع.شارك في المعرض أربعون منتسباً للمركز، وبلغ عدد الأعمال 50 عملاً خطياً وفق خطوط وزخارف متنوعة ومختلفة في المدارس والاتجاهات، وتبدو في المعرض الذي يضم مجموعة من اللوحات الخطية قضية تأكيد الابتكار جلية بالرغم من الالتزام الواضح بميزان القواعد ونواظم كل نوع من الخطوط المستهدفة للتعبير عن التراكيب البصرية اللافتة للخط العربي الأصيل، ولوحظ أن المواضيع المطروحة متعددة بدءاً من الآيات ال

 50%|█████     | 3/6 [00:03<00:03,  1.26s/it]

...



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\n محتوى المقال: \nأبوظبي: «الخليج» أكد عدد من الخبراء والمسؤولين المشاركين في فعاليات جناح جائزة خليفة التربوية أن المعرض أصبح علامة بارزة في مسيرة الفكر والثقافة، وترجم المكانة المرموقة التي تحظى بها أبوظبي كمركز عالمي للإبداع الفكري والإشعاع الحضاري.وشهد جناح الجائزة عدداً من المحاضرات الثقافية وورش العمل التطبيقية التي ترسخ ثقافة التميز لدى مختلف فئات المجتمع.وقالت أمل العفيفي الأمين العام للجائزة: إن البرنامج الثقافي للجائزة شمل محاضرة بعنوان «ثقافة التميز» تحدث فيها الدكتور خالد العبري عضو اللجنة التنفيذية للجائزة حول مسيرة الجائزة منذ انطلاقها في العام 2007 لتشكل بذلك مبادرة وطنية رائدة لغرس ثقافة التميز في الميدان التربوي وتحفيز

 83%|████████▎ | 5/6 [00:06<00:01,  1.17s/it]



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\n محتوى المقال: \nيمكن القول باطمئنان أن شهر رمضان المبارك هو شهر فن الخط العربي، ففي هذا الشهر الكريم تكثر مهرجانات ومعارض الخط في الكثير من البلدان العربية وربما الإسلامية أيضاً، وذلك لاقتران الخط العربي بدلالات دينية روحية إلى جانب سماته الجمالية التي تعلّق بها فنانون تشكيليون غير عرب رأوا في الخط طاقة إبداعية متميزة عن كل خطوط الأبجديات الحية في العالم.لكن رمضان في الإمارات من حيث حجم هذه المعارض يختلف عن أي بلد عربي آخر، فخلال هذا الشهر تنشط وتتفاعل البرامج اليومية المتعلقة بمهرجان الفنون الإسلامية والذي يتميز بأنه (أي المهرجان) فعالية طويلة تستمر الى شهر وأكثر، وحقيقة الأمر أن المهرجان هو مجموعة مهرجانات إذا اعتبرنا أن سلسلة طويل

100%|██████████| 6/6 [00:06<00:00,  1.14s/it]



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\n محتوى المقال: \nيشارك المسرحي الشاب رامي مجدي في مهرجان الشارقة للمسرحيات القصيرة بعرض الواشي لبريخت في ثاني تجربة إخراجية، بعد أن أخرج في الدورة الأولى من المهرجان مسرحية الغرباء لا يشربون القهوة التي قدمته موهبة إخراجية يعوّل عليها في المستقبل، ويقول مجدي عن هذه المشاركة: لقد كانت تجربتي في إخراج (الغرباء لا يشربون القهوة) مفيدة جداً وتعرفت من خلالها إلى كثير من خفايا العملية الإخراجية، خصوصاً طرق التعامل مع النص، وأساليب قيادة الفريق وتوجيه الممثلين وغيرها، وقد كان الانطباع العام لدى النقاد خلال المهرجان أن العمل جيد، ورغم أن العرض استبعد من الجائزة لأنه تجاوز الوقت المحدد للمسرحيات القصيرة، إلا أن تلك التجربة أفادتني كثيراً، وشجع

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data))):
    prompt = data.iloc[i]["prompt"]
    # prompt = prompt[:max_length]
    # result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("الفئة المتوقعة:")[-1].strip()

    pred.append(answer)

100%|██████████| 1000/1000 [19:42<00:00,  1.18s/it]


In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
السياسة,133
6. الرياضة,117
3. الطب,116
التكنولوجيا,99
,97
الثقافة,92
المال,79
4. السياسة,71
7. التكنولوجيا,43


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted']:
  if "الرياضة" in pr:
    nor_pre.append("Sports")
  elif "الصحة" in pr:
    nor_pre.append("Medical")
  elif "الطب" in pr:
    nor_pre.append("Medical")
  elif "الثقافة" in pr:
    nor_pre.append("Culture")
  elif "المال" in pr:
    nor_pre.append("Finance")
  elif "السياسة" in pr:
    nor_pre.append("Politics")
  elif "الدين" in pr:
    nor_pre.append("Religion")
  elif "التكنولوجيا" in pr:
    nor_pre.append("Tech")
  else:
    nor_pre.append("Unclassified")

In [ ]:
pred_zero['Normalized Prediction'] = nor_pre

In [ ]:
pred_zero['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
Politics,208
Tech,142
Sports,135
Medical,123
Culture,122
Finance,106
Unclassified,104
Religion,60


In [ ]:
pred_zero.to_csv('Allam Zero Shot News Classification.xlsx', index = False)

In [ ]:
true = pd.read_excel('sampled_data.xlsx')
y_true = true['subdirectory'].values
print(classification_report(y_true, pred_zero['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

     Culture     0.8525    0.6933    0.7647       150
     Finance     0.8585    0.6067    0.7109       150
     Medical     0.8699    0.7133    0.7839       150
    Politics     0.6538    0.9067    0.7598       150
    Religion     0.9500    0.5700    0.7125       100
      Sports     0.8963    0.8067    0.8491       150
        Tech     0.8380    0.7933    0.8151       150
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.7350      1000
   macro avg     0.7399    0.6362    0.6745      1000
weighted avg     0.8404    0.7350    0.7738      1000



# Pred Few Shot

In [ ]:
data2 = pd.read_excel('News Classification Jais Few Shot.xlsx')

In [ ]:
from tqdm import tqdm
max_length = tokenizer.model_max_length
for i in tqdm(range(len(data2[:6]))):
    prompt = data2.iloc[i]["prompt"]
    prompt = prompt[:max_length]
    result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("الفئة المتوقعة:")[-1].lower()
    print('\n')
    print(result)
    print('\n')
    print(answer)

 17%|█▋        | 1/6 [00:02<00:12,  2.40s/it]



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\nالمثال 1\nمحتوى المقال:\nأكد الفنان عزت أبوعوف، رئيس مهرجان القاهرة السينمائي، إلغاء حفل ختام المهرجان الذي كان من المزمع أن يقام غداً، وذلك بسبب الظروف غير المستقرة التي تمر بها مصر حالياً.ونفى أبوعوف في تصريحات لـ"العربية.نت" سفر الضيوف الأجانب خوفاً مما تشهده مصر من مظاهرات، وما تردد عن أن هذا السفر قد تسبب في عدم إمكانية إقامة حفل الختام، مؤكداً أن وزير الثقافة دكتور محمد صابر عرب هو من أصدر قرار الإلغاء.وعن توزيع الجوائز التي كان من المزمع أن يتم في حفل الختام، شرح أنه سيقام مؤتمر صحافي سيكون بديلاً عن الحفل وسيتم من خلاله إعلان توزيع الجوائز والأفلام الفائزة في هذه الدورة.يُذكر أن حفل افتتاح المهرجان أيضاً تم في هدوء وبعيداً عن 

 33%|███▎      | 2/6 [00:04<00:08,  2.16s/it]



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\nالمثال 1\nمحتوى المقال:\nأكد الفنان عزت أبوعوف، رئيس مهرجان القاهرة السينمائي، إلغاء حفل ختام المهرجان الذي كان من المزمع أن يقام غداً، وذلك بسبب الظروف غير المستقرة التي تمر بها مصر حالياً.ونفى أبوعوف في تصريحات لـ"العربية.نت" سفر الضيوف الأجانب خوفاً مما تشهده مصر من مظاهرات، وما تردد عن أن هذا السفر قد تسبب في عدم إمكانية إقامة حفل الختام، مؤكداً أن وزير الثقافة دكتور محمد صابر عرب هو من أصدر قرار الإلغاء.وعن توزيع الجوائز التي كان من المزمع أن يتم في حفل الختام، شرح أنه سيقام مؤتمر صحافي سيكون بديلاً عن الحفل وسيتم من خلاله إعلان توزيع الجوائز والأفلام الفائزة في هذه الدورة.يُذكر أن حفل افتتاح المهرجان أيضاً تم في هدوء وبعيداً عن 

 50%|█████     | 3/6 [00:06<00:06,  2.16s/it]

...



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\nالمثال 1\nمحتوى المقال:\nأكد الفنان عزت أبوعوف، رئيس مهرجان القاهرة السينمائي، إلغاء حفل ختام المهرجان الذي كان من المزمع أن يقام غداً، وذلك بسبب الظروف غير المستقرة التي تمر بها مصر حالياً.ونفى أبوعوف في تصريحات لـ"العربية.نت" سفر الضيوف الأجانب خوفاً مما تشهده مصر من مظاهرات، وما تردد عن أن هذا السفر قد تسبب في عدم إمكانية إقامة حفل الختام، مؤكداً أن وزير الثقافة دكتور محمد صابر عرب هو من أصدر قرار الإلغاء.وعن توزيع الجوائز التي كان من المزمع أن يتم في حفل الختام، شرح أنه سيقام مؤتمر صحافي سيكون بديلاً عن الحفل وسيتم من خلاله إعلان توزيع الجوائز والأفلام الفائزة في هذه الدورة.يُذكر أن حفل افتتاح المهرجان أيضاً تم في هدوء وبعيداً عن 

 83%|████████▎ | 5/6 [00:10<00:02,  2.09s/it]



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\nالمثال 1\nمحتوى المقال:\nأكد الفنان عزت أبوعوف، رئيس مهرجان القاهرة السينمائي، إلغاء حفل ختام المهرجان الذي كان من المزمع أن يقام غداً، وذلك بسبب الظروف غير المستقرة التي تمر بها مصر حالياً.ونفى أبوعوف في تصريحات لـ"العربية.نت" سفر الضيوف الأجانب خوفاً مما تشهده مصر من مظاهرات، وما تردد عن أن هذا السفر قد تسبب في عدم إمكانية إقامة حفل الختام، مؤكداً أن وزير الثقافة دكتور محمد صابر عرب هو من أصدر قرار الإلغاء.وعن توزيع الجوائز التي كان من المزمع أن يتم في حفل الختام، شرح أنه سيقام مؤتمر صحافي سيكون بديلاً عن الحفل وسيتم من خلاله إعلان توزيع الجوائز والأفلام الفائزة في هذه الدورة.يُذكر أن حفل افتتاح المهرجان أيضاً تم في هدوء وبعيداً عن 

100%|██████████| 6/6 [00:12<00:00,  2.15s/it]



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\nالمثال 1\nمحتوى المقال:\nأكد الفنان عزت أبوعوف، رئيس مهرجان القاهرة السينمائي، إلغاء حفل ختام المهرجان الذي كان من المزمع أن يقام غداً، وذلك بسبب الظروف غير المستقرة التي تمر بها مصر حالياً.ونفى أبوعوف في تصريحات لـ"العربية.نت" سفر الضيوف الأجانب خوفاً مما تشهده مصر من مظاهرات، وما تردد عن أن هذا السفر قد تسبب في عدم إمكانية إقامة حفل الختام، مؤكداً أن وزير الثقافة دكتور محمد صابر عرب هو من أصدر قرار الإلغاء.وعن توزيع الجوائز التي كان من المزمع أن يتم في حفل الختام، شرح أنه سيقام مؤتمر صحافي سيكون بديلاً عن الحفل وسيتم من خلاله إعلان توزيع الجوائز والأفلام الفائزة في هذه الدورة.يُذكر أن حفل افتتاح المهرجان أيضاً تم في هدوء وبعيداً عن 

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data2))):
    prompt = data2.iloc[i]["prompt"]
    # prompt = prompt[:max_length]
    # result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("الفئة المتوقعة:")[-1].strip()

    pred.append(answer)

100%|██████████| 1000/1000 [37:32<00:00,  2.25s/it]


In [ ]:
pred_few = pd.DataFrame()
pred_few['Predicted'] = pred
pred_few['Predicted'].value_counts()

,count
Predicted,
السياسة,113
التكنولوجيا,112
الدين,110
المال,81
الثقافة,81
...,...
المثال 8\nمحتوى المقال:\nأعرب رئيس الوزراء اللبناني سعد,1
المثال 8\nمحتوى المقال:\nأعلنت وزارة الداخلية السعودية عن بدء,1
محتوى المقال:\n\nتحدث المقال عن اكتشاف بحيرة تحت الجليد في القطب,1


In [ ]:
nor_pre = []
for pr in pred_few['Predicted']:
  if "الرياضة" in pr:
    nor_pre.append("Sports")
  elif "الصحة" in pr:
    nor_pre.append("Medical")
  elif "الطب" in pr:
    nor_pre.append("Medical")
  elif "الثقافة" in pr:
    nor_pre.append("Culture")
  elif "المال" in pr:
    nor_pre.append("Finance")
  elif "السياسة" in pr:
    nor_pre.append("Politics")
  elif "الدين" in pr:
    nor_pre.append("Religion")
  elif "التكنولوجيا" in pr:
    nor_pre.append("Tech")
  else:
    nor_pre.append("Unclassified")

In [ ]:
pred_few['Normalized Prediction'] = nor_pre
pred_few['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
Unclassified,289
Religion,116
Politics,116
Tech,112
Medical,108
Finance,98
Culture,86
Sports,75


In [ ]:
pred_few.to_csv('Allam Few Shot News Classification.xlsx', index = False)

In [ ]:
print(classification_report(y_true, pred_few['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

     Culture     0.9302    0.5333    0.6780       150
     Finance     0.8469    0.5533    0.6694       150
     Medical     0.7500    0.5400    0.6279       150
    Politics     0.8190    0.6333    0.7143       150
    Religion     0.7155    0.8300    0.7685       100
      Sports     0.9467    0.4733    0.6311       150
        Tech     0.9286    0.6933    0.7939       150
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.5970      1000
   macro avg     0.7421    0.5321    0.6104      1000
weighted avg     0.8548    0.5970    0.6940      1000



# CoT

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=50,
                temperature=0.2
               )

Device set to use cuda:0


In [ ]:
data3 = pd.read_excel('News Classification Jais CoT.xlsx')

In [ ]:
from tqdm import tqdm
max_length = tokenizer.model_max_length
for i in tqdm(range(len(data3[:20]))):
    prompt = data3.iloc[i]["prompt"]
    prompt = prompt[:max_length]
    result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("الفئة المتوقعة:")[-1].lower()
    print('\n')
    print(result)
    print('\n')
    print(answer)

  5%|▌         | 1/20 [00:05<01:39,  5.25s/it]



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\nاتبع الخطوات التالية عند تحليل محتوى كل مقال:\n\nالخطوة 1: فهم المحتوى الأساسي\nاقرأ المقال بالكامل بعناية. حدد الحدث أو القضية أو الرسالة الرئيسية. قد يكون ذلك خبرًا أو تعليقًا أو تقريرًا يتعلق بمجالات مثل السياسة أو الصحة أو الاقتصاد أو الرياضة أو التكنولوجيا أو الثقافة.\n\nالخطوة 2: تحديد الموضوع السائد\nحدد الموضوع أو الفكرة الرئيسية. ركز على محتوى المقال والمفاهيم المتكررة فيه. تجاهل المواضيع الثانوية أو الجوانب الجانبية التي لا تُعبّر عن التركيز العام للمقال.\n\nالخطوة 3: مطابقة المقال مع فئة\nاختر الفئة الوحيدة التي تتوافق بشكل أفضل مع الموضوع السائد. كن دقيقًا: اختر الفئة الأكثر تحديدًا والأقرب صلة. لا تعتمد على الافتراضات أو 

 10%|█         | 2/20 [00:10<01:36,  5.38s/it]



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\nاتبع الخطوات التالية عند تحليل محتوى كل مقال:\n\nالخطوة 1: فهم المحتوى الأساسي\nاقرأ المقال بالكامل بعناية. حدد الحدث أو القضية أو الرسالة الرئيسية. قد يكون ذلك خبرًا أو تعليقًا أو تقريرًا يتعلق بمجالات مثل السياسة أو الصحة أو الاقتصاد أو الرياضة أو التكنولوجيا أو الثقافة.\n\nالخطوة 2: تحديد الموضوع السائد\nحدد الموضوع أو الفكرة الرئيسية. ركز على محتوى المقال والمفاهيم المتكررة فيه. تجاهل المواضيع الثانوية أو الجوانب الجانبية التي لا تُعبّر عن التركيز العام للمقال.\n\nالخطوة 3: مطابقة المقال مع فئة\nاختر الفئة الوحيدة التي تتوافق بشكل أفضل مع الموضوع السائد. كن دقيقًا: اختر الفئة الأكثر تحديدًا والأقرب صلة. لا تعتمد على الافتراضات أو 

 15%|█▌        | 3/20 [00:16<01:32,  5.47s/it]

...



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\nاتبع الخطوات التالية عند تحليل محتوى كل مقال:\n\nالخطوة 1: فهم المحتوى الأساسي\nاقرأ المقال بالكامل بعناية. حدد الحدث أو القضية أو الرسالة الرئيسية. قد يكون ذلك خبرًا أو تعليقًا أو تقريرًا يتعلق بمجالات مثل السياسة أو الصحة أو الاقتصاد أو الرياضة أو التكنولوجيا أو الثقافة.\n\nالخطوة 2: تحديد الموضوع السائد\nحدد الموضوع أو الفكرة الرئيسية. ركز على محتوى المقال والمفاهيم المتكررة فيه. تجاهل المواضيع الثانوية أو الجوانب الجانبية التي لا تُعبّر عن التركيز العام للمقال.\n\nالخطوة 3: مطابقة المقال مع فئة\nاختر الفئة الوحيدة التي تتوافق بشكل أفضل مع الموضوع السائد. كن دقيقًا: اختر الفئة الأكثر تحديدًا والأقرب صلة. لا تعتمد على الافتراضات أو 

 95%|█████████▌| 19/20 [01:41<00:05,  5.27s/it]



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\nاتبع الخطوات التالية عند تحليل محتوى كل مقال:\n\nالخطوة 1: فهم المحتوى الأساسي\nاقرأ المقال بالكامل بعناية. حدد الحدث أو القضية أو الرسالة الرئيسية. قد يكون ذلك خبرًا أو تعليقًا أو تقريرًا يتعلق بمجالات مثل السياسة أو الصحة أو الاقتصاد أو الرياضة أو التكنولوجيا أو الثقافة.\n\nالخطوة 2: تحديد الموضوع السائد\nحدد الموضوع أو الفكرة الرئيسية. ركز على محتوى المقال والمفاهيم المتكررة فيه. تجاهل المواضيع الثانوية أو الجوانب الجانبية التي لا تُعبّر عن التركيز العام للمقال.\n\nالخطوة 3: مطابقة المقال مع فئة\nاختر الفئة الوحيدة التي تتوافق بشكل أفضل مع الموضوع السائد. كن دقيقًا: اختر الفئة الأكثر تحديدًا والأقرب صلة. لا تعتمد على الافتراضات أو 

100%|██████████| 20/20 [01:47<00:00,  5.36s/it]



[{'generated_text': '\nأنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال: \n1. الثقافة\n2. المال \n3. الطب\n4. السياسة\n5. الدين\n6. الرياضة\n7. التكنولوجيا\n\nاتبع الخطوات التالية عند تحليل محتوى كل مقال:\n\nالخطوة 1: فهم المحتوى الأساسي\nاقرأ المقال بالكامل بعناية. حدد الحدث أو القضية أو الرسالة الرئيسية. قد يكون ذلك خبرًا أو تعليقًا أو تقريرًا يتعلق بمجالات مثل السياسة أو الصحة أو الاقتصاد أو الرياضة أو التكنولوجيا أو الثقافة.\n\nالخطوة 2: تحديد الموضوع السائد\nحدد الموضوع أو الفكرة الرئيسية. ركز على محتوى المقال والمفاهيم المتكررة فيه. تجاهل المواضيع الثانوية أو الجوانب الجانبية التي لا تُعبّر عن التركيز العام للمقال.\n\nالخطوة 3: مطابقة المقال مع فئة\nاختر الفئة الوحيدة التي تتوافق بشكل أفضل مع الموضوع السائد. كن دقيقًا: اختر الفئة الأكثر تحديدًا والأقرب صلة. لا تعتمد على الافتراضات أو 

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(data3))):
    prompt = data3.iloc[i]["prompt"]
    # prompt = prompt[:max_length]
    # result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("الفئة المتوقعة:")[-1].strip()

    pred.append(answer)

100%|██████████| 1000/1000 [1:28:14<00:00,  5.29s/it]


In [ ]:
pred_cot = pd.DataFrame()
pred_cot['Predicted'] = pred
pred_cot['Predicted'].value_counts()

,count
Predicted,
السياسة,132
الرياضة,112
التكنولوجيا,109
المال,86
الطب,83
...,...
"الدين\n\n\n Article Content: \nكمال قبيسي\nخبر علمي ساخن تحدثت العربية بشأنه إلى عالم مصري في وكالة الفضاء الأمريكية ""ناسا"" ويأتي هذه المرة من حيث لا شيء سوى الجليد الأبدي، ففي الخميس الماضي أعلن علماء محطة علمية روسية بالقطب الجنوبي عن بلوغهم عتبة ما حفروا 20 سنة للوصول إليه، وهي بحيرة تحت ثلج متلبد فوقها بسماكة 3600 متر، وما زالت مياهها كما كانت منذ 20 مليون عام.\n\n\n\n\n\nلم ير أحد بحيرة ""فوستوك"" قبل الآن، ولا لمس مياهها أيضا، لأنها تحولت الى ""كبسولة زمن"" نامت في ظلام تام خيم عليها منذ حدثت تغييرات حاسمة بمناخ الأرض فغطت الثلوج مياهها بما كان فيها من كائنات سبقت وجود الإنسان على الأرض بأكثر من 400 ألف عام، طبقا لما يؤكده علماء الأحياء. وتقع البحيرة بمنطقة من القطب الجنوبي ترتفع 3500 متر عن سطح البحر، وهي أكبر 150 تجمعا للماء تحت الثلوج، وتم اكتشافها في 1996 برصد من أقمار اصطناعية ومجسات استشعار ورادارات بالمسح الاسترجاعي للضوء وجدتها تحت منطقة سبق للعلماء الروس أن بدأوا في 1991 بحفر بئر في ثلوجها لتجاربهم، فتابعوا حفره بعد اكتشافها بهدف اختراق سقفها الجليدي والوصول إلى مياهها للتعرف الى ما تحتويه. ومن التفاصيل عن البحيرة البالغة مساحتها 15690 كيلومتر مربع، أن مياهها العميقة 800 متر هي أنظف وأنقى ما يمكن للإنسان أن يعرفه، وهي عذبة ومشبعة بأوكسيجين يزيد 50 مرة عن الموجود في الماء العادي، وتكفي لسد حاجات دولة كالسعودية مثلا طوال 200 عام، لكن ما يرغب به العلماء هو معرفة ما إذا كانت تحتوي على الأهم، وهو أي نوع من الحياة يعتقدون بأنه مختلف عن التقليدي المعروف.\n\n\n\nماء على الأرض يؤكد وجود الحياة في الفضاء\n\n\nومعظم التقارير عن ""فوستوك"" الأكبر مساحة من لبنان بأكثر من 5000 كيلومتر مربع، تنتهي بالتفاؤل في أن الوصول إلى مياهها سيحمل للإنسان مفاجآت، حتى ولو كانت طبقات الجليد المتراكمة فوق مياهها منذ ملايين السنين قضت على ما كان فيها من كائنات، لأن رواسبها بقيت في قعرها. أما إذا اتضح أن الحياة مستمرة فيها، ولو في بكتيريا بدائية ""فذلك سيؤكد إمكانية وجودها على الأقل في المريخ، أو قمر ""أوروبا"" الدائر حول المشتري، والمرشح لتبرعم الحياة بمعزل عن حرارة الشمس"" بحسب ما قال الدكتور عصام حجي، وهو رئيس في ""ناسا"" لفريق ناشط ببرنامج يستخدم قمراً اصطناعيا حول الأرض للبحث عن المياه الجوفية في قطبي الأرض، كما في صحاريها، وتشارك فيه الكويت مع إيطاليا إضافة لوكالة الفضاء الأمريكية. وشرح الدكتور حجي عبر الهاتف من كاليفورنيا حيث يقيم وينشط في معمل محركات الدفع الصاروخي التابع لناسا، أن قمر ""أوروبا"" شبيه بالقطب الجنوبي، فسطحه ملبد بجليد سماكته بالكيلومترات ويعتقدون بأن في أسفله تجمعات مائية شبيهة بمياه ""فوستوك"" المعزولة عن البيئة التقليدية وضوء الشمس وحرارتها منذ ملايين السنين. وذكر أن العثور على أي نوع من البكتيريا في ""فوستوك"" لن يدلنا إلى ما كانت عليه طبيعة الحياة قبل 20 مليون سنة على الأرض فقط، بل سيؤكد أن بإمكانها أن تتبرعم وتنشأ ""في مظهر لا نعرفه، سواء في القطب الجنوبي أو أسفل ثلوج قمر أوروبا، أو في المريخ، بل وفي كواكب خارج المجموعة الشمسية"" كما قال. ولأن الحياة ""لا تنشأ من دون أشعة الشمس وحرارتها، فإن العثور على أي مظهر لها في بحيرة ""فوستوك"" المعزولة عن العالم والأشعة الشمسية سيفتح نافذة للإنسان مهمة يطل منها على نوع جديد ومختلف من الحياة لم نعرفها من قبل"" وهو ما ينتظره الكتور حجي ويتفاءل بوجوده.",1
"الدين\n\n\n Article Content: \nأعلنت شركة شاومي الصينية اليوم عن وصول عدد طلبات التسجيل المسبق للحصول على أحدث هواتفها الرائدة Xiaomi Mi 5 من داخل الصين إلى 16.8 مليون طلب. وتشكل كمية الطلبات الحالية رقماً قياسياً جديداً للشركة في عضون أسبوع من الكشف عنه، وتنوي شاومي توفير الهاتف للبيع في عدد من الدول الآخرى قريباً. وكانت الشركة قامت بالإعلان عن الهاتف منذ نحو أسبوع تقريباً ضمن مؤتمر الجوال العالمي ببرشلونة على أن يبدأ بيعه للمستخدمين في الصين بتاريخ 1 مارس (آذار). ويأتي الهاتف بالمواصفات التالية: 1- بشاشة عرض قياس 5.15 إنش وبدقة 1920×1080 بكسل. 2- بذاكرة عشوائية بخيارين إما 3 أو 4 غيغابايت، وثلاثة خيارات سعة تخزين داخلية إما 32 أو 64 أو 128 غيغابايت. 3- يعمل بنظام ""أندرويد مارشميلو-6.6"". 4- بمعالج ""سناب دراغون 820"" وبطارية غير قابلة للإزالة بسعة 3000 ميللي أمبير. 5- بكاميرا خلفية دقة 16 غيغابايت، ومثبت بصري ""أو آي إس"" وتقنية للتركيز التلقائي، وكاميرا أمامية بدقة 4 ميغابكسل. 6- يدعم شريحتي اتصال ""سيم""، وشبكات الجيل الرابع ""إل تي إي"

In [ ]:
nor_pre = []
for pr in pred_cot['Predicted']:
  if "الرياضة" in pr:
    nor_pre.append("Sports")
  elif "الصحة" in pr:
    nor_pre.append("Medical")
  elif "الطب" in pr:
    nor_pre.append("Medical")
  elif "الثقافة" in pr:
    nor_pre.append("Culture")
  elif "المال" in pr:
    nor_pre.append("Finance")
  elif "السياسة" in pr:
    nor_pre.append("Politics")
  elif "الدين" in pr:
    nor_pre.append("Religion")
  elif "التكنولوجيا" in pr:
    nor_pre.append("Tech")
  else:
    nor_pre.append("Unclassified")

In [ ]:
pred_cot['Normalized Prediction'] = nor_pre
pred_cot['Normalized Prediction'].value_counts()

,count
Normalized Prediction,
Politics,170
Tech,136
Sports,136
Religion,130
Medical,126
Finance,120
Unclassified,95
Culture,87


In [ ]:
pred_cot.to_csv('Allam CoT News Classification.xlsx', index = False)

In [ ]:
print(classification_report(y_true, pred_cot['Normalized Prediction'].values, digits = 4))

              precision    recall  f1-score   support

     Culture     0.9425    0.5467    0.6920       150
     Finance     0.8750    0.7000    0.7778       150
     Medical     0.9286    0.7800    0.8478       150
    Politics     0.7765    0.8800    0.8250       150
    Religion     0.6385    0.8300    0.7217       100
      Sports     0.9779    0.8867    0.9301       150
        Tech     0.8456    0.7667    0.8042       150
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.7670      1000
   macro avg     0.7481    0.6738    0.6998      1000
weighted avg     0.8658    0.7670    0.8037      1000

